In [ ]:
CST4225 – Coursework 1 (30%)
Title: Airbnb Rental Income & Listing Success Prediction
Platform: Dataiku (Python if needed)
Deadline: 6
th December 2026
Submission: Upload to MyLearning → Link will be shared
Word Limit: 2000 words (Main Questions -excluding code, tables, figures etc)




In [5]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd

path = "F:/Users/MSR/Downloads/neighbourhoods/listings.csv"
df = pd.read_csv(path)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
print(df.shape)
print(df.info())

print(df.head())


F:\Users\MSR\AppData\Local\Temp\ipykernel_17544\518413514.py:7: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


(87946, 75)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87946 entries, 0 to 87945
Data columns (total 75 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            87946 non-null  int64  
 1   listing_url                                   87946 non-null  object 
 2   scrape_id                                     87946 non-null  float64
 3   last_scraped                                  87946 non-null  object 
 4   source                                        87946 non-null  object 
 5   name                                          87946 non-null  object 
 6   description                                   86679 non-null  object 
 7   neighborhood_overview                         47190 non-null  object 
 8   picture_url                                   87943 non-null  object 
 9   host_id                                       879

In [2]:
Overview
In this challenge, you will 
        build: a predictive model to estimate the success of Airbnb property listings. 
                Success: Represented by a categorical variable (e.g., Good / Bad listing) — 
                            derived using own formula based on key indicators such as price,occupancy, and reviews.

Work through the full data science lifecycle for this objective: 
domain familiarisation, 

feature engineering, 

model development, 

evaluation, and 

interpretation

— while maintaining proper documentation and a reproducible workflow in Dataiku.
Python may be used where needed.

In addition, you should propose: 
        one additional data science use case that could be developed using the Airbnb datasets. 
        For this second objective:

                Required to explain what the proposed use case is, 
                                    why it is useful, 
                                    what data would be required,
                                    and how it could potentially be developed. 
                                    (You are not required to execute the full lifecycle for Objective 2.)


The provided dataset contains a wide range of attributes related to Airbnb listings, 
        such as property type, 
                location, 
                number of bedrooms, 
                amenities, 
                rental price, and more.

Additionally, a second dataset provides information on Airbnb
        customer reviews, which could be used as a proxy to measure how many tourists rent each Airbnb listing. 

Finally,a third dataset provides information on 
        the different neighbourhoods of London.
The dataset is uploaded on MyLearning, and can also be downloaded from:
http://insideairbnb.com/get-the-data



SyntaxError: invalid character '—' (U+2014) (1952867457.py, line 4)

In [ ]:
Based on the paper by Negi and Tripathi (2023) and the 75-column Airbnb dataset you provided, 
I have mapped the paper's eight research themes to specific, calculable KPIs that address real business problems. 
These KPIs are designed to be implemented in Dataiku using the available columns.

### 🎯 Mapping Paper Themes to Dataiku-Ready KPIs

The paper segments Airbnb research into eight areas: **consumer behavior, 
                                                        host behavior, host–guest relationship, trust, 
                                                        dominant theories, regulation, Airbnb vs. hotels, and macro impacts**.
                            Here are the corresponding KPIs you can build:

| Paper Theme (Business Problem)              | Proposed KPI               | Data Columns to Use in Dataiku | Business Rationale                                                       |
| :------------------------------------------ | :------------------------- | :----------------------------  | :----------------------------------------------------------------------- |
| **Consumer Behavior**                       |  **Value-for-Money Score** | `review_scores_value`, `price` | Identifies listings that guests perceive as high-value,
|    (Understanding guest preferences & value)|                            |  `accommodates`                |  which is crucial for pricing strategy and competitive positioning.       |
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Host Behavior**                           | **Professional Host Ratio**|`calculated_host_listings_count`| Segments the market. Professional hosts (e.g., >5 listings) 
|(Identifying professional vs. casual hosts)  | __________________________ | `host_listings_count`          | may have different operational needs and impact than single-listing hosts. |___________________
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Host-Guest Relationship** (Service qualit | **Composite Service        | `review_scores_cleanliness`,   | Averages key service metrics to pinpoint operational strengths and weaknesses, |
| (Service quality & satisfaction)            |   Quality Index**          | `review_scores_checkin`,       | directly impacting guest satisfaction and repeat business. |
|____________________________________________ |                            |  `review_scores_communication` |_______________________________________________________________________
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Trust in Airbnb**                         |**Superhost Trust Premium** | `host_is_superhost`,          | Quantifies the tangible benefits (e.g., higher ratings, more reviews) 
|(Platform credibility & safety)              | __________________________ | `review_scores_rating`,        | of the "Superhost" badge, validating the platform's trust mechanisms. |_______________________________________________________________________
|____________________________________________ | __________________________ | `number_of_reviews`            |_______________________________________________________________________
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Airbnb Regulation**                       | **License Compliance Rate**| `license` (non-null count)     | Measures the proportion of listings with a license.                     |
|(Compliance & policy impact)                 |____________________________|________________________________| Essential for policymakers and the platform to monitor regulatory adherence. _______________________________________________________________________
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Airbnb vs. Hotels**                       | **Entire Home vs.          | `room_type`, `property_type`   | Tracks the market share of entire homes/apartments versus traditional hotel-like rooms, 
| (Competitive landscape)                     | Hotel Room Share**         |________________________________| indicating the platform's disruptive impact.                             |
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Macro Impacts**                           | **Listing Density per      | `neighbourhood_cleansed`       | A proxy for tourism pressure and potential housing strain in specific areas, 
|  (Housing availability & tourism density) __|    Neighborhood**          |  (count of `id`)               | useful for urban planning and sustainable tourism analysis. |______________________________________________________________________
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________
| **Host Behavior**                           | **Average Minimum Stay**   | `minimum_nights`,              | Indicates host strategy (short-term vs. long-term rental) and 
| (Booking flexibility & demand)              |____________________________| `maximum_nights`               | can be correlated with `availability_365` to understand booking constraints. |
|_____________________________________________|____________________________|________________________________|_______________________________________________________________________

### 💡 Implementation Notes for Dataiku

- **Data Cleaning:** The `price` column is an `object` (likely contains "$" and ","). \
    You must clean and convert it to a `float` in a Prepare recipe before calculating any financial KPIs.
- **Handling Missing Values:** Columns like `license` (only 1 non-null) or `bathrooms` (0 non-null) are unusable. 
    For `review_scores`, you can filter out missing values or impute them based on the neighborhood average, depending on your analysis goal.
- **Derived Metrics:** For the "Composite Service Quality Index," use a Formula or Python recipe to create a new column averaging the three review scores. 
    For "Listing Density," use a Group recipe to count listings per `neighbourhood_cleansed`.

Would you like me to elaborate on the Dataiku recipe steps (e.g., Prepare, Group, Window) for calculating any of these specific KPIs?

In [ ]:
Template for Report Preparation
Important:
The entire report must be written in this pattern.

For each question, 
    all written explanations and interpretations appear first,
    followed by all Dataiku screenshots, Python code segments, model outputs, charts, or other technical evidence at the end.

    Technical evidence must be clearly referenced in the report text as Q1A, Q1B, etc.
Example:
To explore the distribution of property types across the dataset, I created a bar chart
illustrating the most common Airbnb property categories (see Q1A visual). The resulting
visualization highlights that entire apartments and private rooms make up over 80% of
listings (see Q1B Dataiku output).
Where Python is used, relevant code should also be placed at the end of the respective
question.

In [ ]:
25% Includes                                                                                                                                               (500 words)
    problem setup, dependent variable logic, and justification of formula.
    Data processing & feature engineering

In [ ]:
Q1. Domain Knowledge Building (1 paragraph)
    What process did you follow to understand Airbnb’s business model, 
    host success metrics, or pricing factors?
    1-2 para max
Suggested Dataiku evidence:
If you performed any initial profiling, charts, distributions, or exploratory analysis in
Dataiku, include relevant screenshots at the end as Q1A, Q1B, etc.

In [ ]:
Q2. Data Sourcing
    Did you use any additional dataset beyond the given one? (Yes/No)
    If Yes, describe:
    • What data you sourced (2–3 lines)
    • The exact reference or link(s)
    • How you merged the data (3 bullet points: join keys, merge method, checks)
    If possible, give a screenshot of how merged data looks like (or data structure)

In [ ]:
Q3. Dependent Variable Formulation
    (a) Describe your formula for defining Good/Bad listing (e.g., based on revenue, rating,
    occupancy). Explain the logic in one paragraph.
    (b) Show the formula used to generate this dependent variable.
    This can be created using:
    • Dataiku formula
    • Prepare recipe
    • Python, where needed
    Include the formula or relevant screenshot at the end.


In [ ]:

1. Build Domain Knowledge & Frame the Problem: 
    Begin by understanding the context and dynamics of the Airbnb marketplace. 
    Strengthen your domain knowledge using available datasets and reliable sources to 
    identify what defines success in property listings. 
    Use this understanding to guide your data exploration and modeling direction throughout the challenge.


2. Define & Construct the Success Metric: 
    Establish a clear and measurable indicator of success based on data-driven logic (Good/Bad listing). 
    The dependent variable should capture listing performance meaningfully and 
    be justified using available attributes. 
    (Ensure your success metric aligns with business understanding and can be reproduced consistently)

In [ ]:
For the Airbnb dataset you provided, 
the most urgent and critical step is to **clean the `price` column**,
because it is of type `object` and likely contains currency symbols and thousands separators, 
making it impossible to perform any numerical calculations directly. 
    Below, using the **Value-for-Money Score** as an example, I will break down the complete Dataiku recipe workflow in detail.

### Core KPI Implementation: Value-for-Money Score (Value-for-Money Score)

**Business Objective**: Calculate the value-for-money score perceived by guests for each listing, 
                        in order to identify listings with "high ratings and low prices," providing a basis for pricing strategy and competitive analysis.

                                      Normalized `review_scores_value` (guest rating) 
**Logic**: Value-for-Money Score = _____________________________________________________________

                                          Normalized `price` (per-person price). 
                                        
Before calculation, `price` must be cleaned and converted to a numeric type, and the per-person price must be calculated based on `accommodates` (number of guests).

---

### Detailed Steps

#### Step 1: Create a Prepare Recipe to Clean the `price` Column

In the Flow, select the Airbnb dataset and create a new **Prepare** recipe.

**1. Remove Currency Symbols and Thousands Separators**
        Click on the header of the `price` column and select **"Remove characters"** from the dropdown menu, or search for the processor in the Script panel. 
                                                      Since the `price` column format is typically like `"$1,200.00"`, you need to remove `$` and `,`. 
        Dataiku's processor supports batch removal of specified characters.

**2. Convert to Numeric Type**
        After cleaning the symbols, change the column's storage type to `double`. 
        In the Prepare recipe, you can directly click on the type identifier below the column name and select `double`. 
            If the conversion fails (shows red), it means there are residual non-numeric characters, and you need to return to the previous step to check.

**3. Rename to `price_numeric`**
To avoid confusion with the original column, rename the cleaned column to `price_numeric`.

#### Step 2: Create a "Per-Person Price" Field

Still in the same Prepare recipe, add a **Formula** step (Formula processor).

Use Dataiku's formula language to create a new column `price_per_person`:
```
price_numeric / accommodates
```
The `accommodates` column is of type `int64` and has no missing values, so it can be used directly for division. 
    After creating the formula step, Dataiku will automatically calculate this new column.

#### Step 3: Create the Value-for-Money Score (Using the Window Recipe for Normalization)

Since `review_scores_value` (usually 1–5) and `price_per_person` (range may be very wide) are on different scales, 
    they cannot be directly divided. Before dividing, they need to be **normalized (Min-Max normalization)** to the [0, 1] range.

**Alternative Approach: Use the Window Recipe to Calculate Min/Max and Normalize**

1. Create a **Window** recipe with the output of the previous Prepare recipe as input.
2. In the Window recipe, set **no partitioning** (i.e., compute over the entire dataset), and add two window fields:
   - `min_price_per_person` = `min(price_per_person)`
   - `max_price_per_person` = `max(price_per_person)`
   - Similarly, compute the min and max for `review_scores_value`.
3. After outputting the results, create another **Prepare** recipe and use the **Formula** processor to calculate the normalized score:
   ```
   (review_scores_value - min_review_scores_value) / (max_review_scores_value - min_review_scores_value)
   ```
   And similarly normalize `price_per_person`.
4. Finally, calculate the value-for-money score:
   ```
   normalized_review_value / normalized_price_per_person
   ```
   **Note**: If `normalized_price_per_person` is 0 (i.e., the minimum price listing), a very small constant (e.g., 0.001) needs to be added to the denominator to avoid division by zero.

**Simplified Alternative**: If you do not want to use the Window recipe, you can also first use the **Group** recipe (no grouping key, i.e., global aggregation) to calculate the global min and max values, 
                            and then join them back to the original dataset for normalization. 
                            This approach is more intuitive but requires one additional join operation.

#### Step 4: Handle Missing Values (review_scores_value has approximately 22,000 Missing Values)

The `review_scores_value` column has 64,820 non-null values (approximately 26% missing). Before normalization, you need to decide the handling strategy:
- **Filter out**: If the analysis only focuses on listings with reviews, add a filter step in the Prepare recipe: keep rows where `review_scores_value` is not null.
- **Impute**: Use the **Fill missing values** processor to fill with 0 or with the median. However, for "value-for-money score," filling with 0 would distort the meaning, so filtering is recommended.

#### Step 5: Output and Verify

Run the Prepare recipe and check whether the distribution of the output `value_for_money_score` is reasonable. 

    You can use Dataiku's **Analyze** feature to view the histogram of this column, confirming that no outliers 
    (e.g., negative values or extreme large values) have been generated due to denominator issues.

---

### Additional Notes

- **Reusability of `price` cleaning**: The output of this step (the cleaned `price_numeric`) can be reused for all subsequent KPIs involving price, 
                                                                             such as "Entire Home vs. Hotel Room share" or "average price per neighborhood." 
                                                                             It is recommended to save this cleaning step as a separate recipe or a shared intermediate dataset.
- **Regarding `license` and `bathrooms`**: These two columns have 1 or 0 non-null values, respectively, and cannot be used to calculate any KPIs.
                                            It is recommended to exclude them in the Prepare recipe or mark them as "unused" in the schema, to avoid subsequent steps attempting to use them by mistake.
- **Engine selection**: For a dataset of 87,946 rows, the default DSS engine is fully capable. 
                        If you have already connected to a SQL database (e.g., Snowflake), you can specify the SQL engine when creating the recipe to leverage pushdown computation and improve performance.

If you need me to further break down the recipe steps for another KPI (such as "Superhost Trust Premium" or "Listing Density"), please let me know.

In [ ]:
25% Quality of preprocessing, transformations, and rationale behind feature construction.                                                                                (500 words)
    Model building & algorithm implementation

In [ ]:
Q4. Data Preprocessing (Up to 10 Steps)
    For each step, describe what you did and why it was necessary. Add Dataiku/code
    snippets or screenshots wherever applicable at the end of this block (Q4).
    Possible steps include handling missing values, scaling, encoding, filtering,
    transformation, etc.

In [ ]:
Q5. Hypotheses Formulation
    For each hypothesis, provide:
    • Hypothesis statement
    • How you arrived at it (pivots, visualizations, domain knowledge, literature)
    • Four-dimensional reasoning: 
        variable, hypothesis, data availability, data quality.

In [ ]:
3. Data Pre-processing & Feature Engineering: 
    Clean and prepare your data systematically to improve model quality. 
    Apply relevant transformations, 
    handle missing values, and 
    create features that enhance prediction capability. 
    Each preprocessing decision should be purposeful and rooted in analytical reasoning.

4. Model Building & Evaluation: 
    Develop and refine your predictive model using sound analytical methods. 
    Experiment with suitable algorithms, 
    adjust variables, and 
    validate model performance using appropriate evaluation metrics. 
    goal is to achieve a balance between accuracy, 
                                        interpretability, and
                                        generalisability (your Dataiku workflow and/or Python code should reflect this).

In [ ]:
Based on the Airbnb dataset you provided earlier and the Dataiku platform environment, the following is a complete model building plan centered on the core business problem of 
**"predicting whether a host becomes a Superhost (`host_is_superhost`)"**. 
    This plan directly addresses the three modules you requested: model building, business translation, and critical reflection.

---

### 1. Model Building Process

**Model Selection and Rationale**
Logistic regression and decision trees are chosen for parallel modeling. 
    Logistic regression is used to **interpret coefficients and quantify marginal effects** 
(for example, "how many percentage points does each 0.1 increase in rating raise the probability of becoming a Superhost"), 
                                  directly addressing business concerns; the decision tree is used to **capture nonlinear relationships and generate rule-based insights** 
(for example, "what threshold combinations of response rate and rating can achieve a 70% probability of becoming a Superhost"). 
                                  The academic literature also confirms that logistic regression is the standard method for Superhost prediction research (following Gunter's model framework), 
                                    while decision trees perform well in similar Airbnb classification tasks with strong interpretability.

**Training/Testing Strategy**
In Dataiku's visual ML interface, a **random 80/20 split** is used, with 80% of the data for training and 20% held out for testing. 
        Considering the potential class imbalance in the Superhost label (typically Superhosts account for about 30-40%), 
        **stratified sampling** is enabled during splitting to ensure consistent class proportions in both training and test sets. 
        The optimization metric is selected as **AUC (Area Under the ROC Curve)**, as it is insensitive to class imbalance and measures the model's ability to rank positive examples (Superhosts) higher.

**Tuning Approach**
- **Logistic Regression**: L2 regularization is used to prevent overfitting, with the regularization strength C tuned via grid search (candidate values: 0.01, 0.1, 1, 10). 
    Dataiku's GLM binary classification module defaults to binomial distribution + logit link function, which is standard logistic regression.
- **Decision Tree**: Maximum depth limited to 4-5 levels (to maintain interpretability), minimum samples per leaf set to 50 (to prevent overfitting to noise), and the splitting criterion is Gini impurity. 
    Pruning is achieved by controlling depth and leaf size.
- **Number of iterations**: Logistic regression does not involve "iterations" in the conventional sense; Dataiku's solver automatically converges. 
    Decision tree training is deterministic and requires no iteration count setting. If random forest is subsequently attempted for performance comparison, 100-200 trees can be set.

**Variable Selection and Encoding Processing**
Initial features include: `host_response_rate`, `host_acceptance_rate`, `number_of_reviews`, `review_scores_rating` (and sub-scores), 
    `calculated_host_listings_count`, `minimum_nights`, `availability_365`, `instant_bookable`, `room_type`, `property_type`, `accommodates`, `bedrooms`, `beds`.

Key processing steps in Dataiku:
- **`host_response_rate` and `host_acceptance_rate`**: These are `object` type columns containing percentage signs (e.g., "95%"). 
                                                        Use the **Prepare recipe** to remove "%" and convert to `double` (0-100 or 0-1).
- **`host_is_superhost`**: Convert from `t`/`f` to `1`/`0` as the target variable.
- **Categorical variables** (`room_type`, `property_type`): Use Dataiku's default **Dummy encoding**; for `property_type` with numerous categories, 
                                                            set "minimum sample size per category" to merge rare categories.
- **Missing value handling**: `review_scores_*` has approximately 26% missing values. 
                                Since missing reviews may itself carry information (new listings or low activity), 
                                use **"Treat missing as a separate category"** rather than deleting rows, preserving sample size.
- **Excluded variables**: `license` (only 1 non-null value), `bathrooms` (all null), and ID/URL type fields.

---

### 2. Model Interpretation and Business Translation

**Most Influential Variables (Expected Based on Literature and Data Structure)**

**Variable 1: Overall Rating (`review_scores_rating`)**
In the Italian market study, overall rating is the variable with the **largest marginal effect** on becoming a Superhost—an AME of 0.39-0.64 across four cities, 
    meaning that if all other conditions remain unchanged, increasing the rating by one unit can raise the probability of becoming a Superhost by 39-64 percentage points. 
        **Business translation**: Rating is the "hard currency" for Superhost qualification.
            Hosts should prioritize investing resources in the dimensions that most affect overall rating—based on the sub-score structure of this dataset, `review_scores_cleanliness` and `review_scores_checkin` typically have the highest weights. This is more effective than blindly increasing response speed.

**Variable 2: Response Rate (`host_response_rate`)**
Studies rank it as the third most important variable, and SAS modeling results also identify `host_response_rate` as one of the most important features. 
        **Business translation**: For hosts, response rate is the metric most easily improved in the short term. 
        Setting automatic reply templates and enabling mobile notifications can rapidly increase response rate from 80% to 95%+, with a direct and quantifiable return on Superhost qualification.
        For platform operations teams, hosts with response rates below a threshold (e.g., <90%) are high-risk churn targets, suitable for targeted intervention.

**Variable 3: Number of Listings Under the Host (`calculated_host_listings_count`)**
The SAS study found that `number of host listings` is one of the most important variables in the final model. However, its direction requires careful interpretation: professional hosts with multiple listings may have operational advantages (systematic management, faster responses), but may also be penalized by Airbnb for "excessive commercialization." **Business translation**: Single-listing hosts need to compensate for their lack of scale with "high-quality service + high rating"; multi-listing hosts should focus on maintaining consistent experience across all listings, as a poor listing may drag down the host's overall Superhost status.

**Variable 4: Number of Reviews (`number_of_reviews`)**
Both the SAS model and Italian studies confirm its importance. **Business translation**: Review count is a proxy variable for "cumulative trust." New listings (review count < 10) have difficulty obtaining Superhost status regardless of quality. The platform should provide a "new host growth pathway" that does not rely on review accumulation but instead validates service quality through other signals (such as identity verification + first-order guarantee).

**Decision Tree Business Rules (Example)**
If the decision tree outputs a rule such as: `review_scores_rating > 4.8 AND host_response_rate > 95% AND number_of_reviews > 50` → Superhost probability > 80%. This can be directly packaged as a **"Superhost Readiness Self-Check Tool"**: hosts input their data and receive a clear gap list ("You need 20 more reviews" or "Your response rate needs to increase by 8 percentage points"). This is more actionable than the black-box probabilities of logistic regression.

---

### 3. Critical Reflection and Product Development

**What Went Well**
- **Business interpretability**: Logistic regression coefficient direction and marginal effects directly translate into host action recommendations, avoiding the "high accuracy but unexplainable" dilemma.
- **Feature engineering leverage**: Converting percentage-formatted columns like `host_response_rate` in the Prepare recipe is a necessary prerequisite for model validity; Dataiku's Prepare visual processor makes this step traceable and replicable.
- **Addressing missing values**: Using "missing as a category" rather than deleting rows preserves the possibility for new listings to be scored, avoiding systematic bias.

**Areas for Improvement**
- **Class imbalance**: If Superhosts account for less than 25%, the default 0.5 threshold for logistic regression may bias toward predicting "non-Superhost." The threshold should be adjusted based on the precision-recall curve in Dataiku's model results interface, or **class weights** should be set (balanced mode).
- **Temporal validation missing**: The current random split ignores the time dimension. Hosts who became Superhosts in 2018 face different market conditions than those in 2022. Dataiku supports splitting by time variable (e.g., `host_since`), which better simulates "predicting the future with the past" for production deployment.
- **Text information not utilized**: The dataset contains `description`, `amenities`, and other text fields, which contain signals about listing quality and host professionalism. These were not included in the model in this round—this is a clear area for improvement.

**Scalability and Productization Path**
- **Automated scoring pipeline**: Package the Prepare cleaning steps (price and percentage column handling) as a **shared recipe** for reuse by other KPIs. Create a **scoring recipe** that automatically scores new scraped data using the trained logistic regression model, outputting "Superhost probability" for each listing.
- **Productization direction**: The model output can be packaged as an **"Airbnb Host Growth Assistant"** —hosts input their listing data and receive: (1) current Superhost probability; (2) the top three variables with the largest gaps; (3) specific improvement actions for each variable (e.g., "Enable Instant Book to reduce cancellation rate," "Set up automatic replies to increase response rate"). This transforms the model from a "prediction tool" into a "decision tool."
- **Limitations**: (1) The model is based on cross-sectional data and cannot capture dynamic changes in host behavior; (2) the Superhost standards themselves change over time with platform policy adjustments, and the model requires periodic retraining; (3) `license` field has a non-null rate of nearly zero, making it impossible to analyze the impact of regulation on Superhost status—this is a key gap for future research given the academic attention to Airbnb regulation.

**Future Opportunities**
Introducing **survival analysis** (time from host registration to first Superhost qualification) could answer "how long it takes to become a Superhost," which is more aligned with host growth needs than binary classification. Alternatively, using the text topics of `amenities` as features could identify "which amenity combinations most strongly correlate with Superhost status" (e.g., workspace vs. parking space), providing data support for hosts' listing optimization.

---

**Appendix Note**: Model output screenshots (confusion matrix, ROC curve, variable importance ranking) should be stored separately in the appendix, not in the main text, to maintain fluency in the main narrative. In Dataiku, screenshots can be taken directly from the "Model Results" page and annotated with the model name and date.

In [ ]:
25% Suitability of algorithm, training/testing rigor, evaluation strategy, reproducibility.                                                                              (500 words)
    Interpretation & lifecycle closure

In [ ]:
Q6. Model Building Process
    Describe your modeling approach (logistic regression, decision tree or both). 
    Explain your training/testing strategy, number of iterations, tuning approach, or variable selection logic. 
    Mention any changes made to independent variables or encoding.
    Present model output screenshots and extra ones separately in the appendix.


In [ ]:
5. Model Interpretation & Business Translation: 
    Focus on interpreting your model outcomes to generate actionable insights. 
    Identify the most influential variables and 
    explain their significance in business terms. 
    Emphasize clarity and practical relevance over purely statistical descriptions.

6. Critical Reflection & Product Development: 
    Evaluate the overall process critically, 
    highlighting what worked well and what could be improved. 
    Discuss potential scalability, automation, or productisation of your solution. 
    Reflecting on limitations and future opportunities demonstrates applied understanding and innovation.

In [ ]:
Based on the modeling plan established earlier (predicting `host_is_superhost`), combined with the actual statistical patterns of the Airbnb dataset and Airbnb's official Superhost assessment rules, below is the complete explanation of model interpretation and critical conclusions. The core judgment is: **the model's predictions may "go both ways"—depending on the quality of feature engineering and the randomness of the data split, the model may either accurately capture the "rating-driven" mechanism or be misled by "response rate illusion" or missing value patterns**. The following paragraphs first present the best-case and worst-case scenario descriptions, then expand on business interpretation and critical conclusions.

---

## I. Best-Case vs. Worst-Case Scenario: What the Model Output and Data Values Might Look Like

### Best-Case Scenario

The model (logistic regression) achieves an **AUC around 0.82–0.88** on the test set, and the coefficient directions align with the core findings of the academic literature. `review_scores_rating` shows the strongest positive effect: with other variables held constant, each 0.1 increase in rating raises the probability of becoming a Superhost by **8–12 percentage points** (marginal effect). `host_response_rate` follows closely, with a coefficient significantly positive but of smaller magnitude—raising response rate from 80% to 95% increases Superhost probability by approximately **5–7 percentage points**. `number_of_reviews` shows a diminishing marginal contribution (log-transformed), with new listings (reviews < 10) having extremely low predicted probabilities regardless of other features. Decision tree outputs clean business rules, such as: `review_scores_rating > 4.85 AND host_response_rate > 0.95 AND number_of_reviews > 40` → predicted Superhost probability > 75%. Confusion matrix shows recall for the positive class (Superhost) around 0.70–0.78, with precision around 0.72–0.80, and the threshold adjusted to around 0.45 (rather than the default 0.5) based on the precision-recall curve. In this scenario, the model's business value lies in: **providing hosts with a quantifiable "Superhost readiness gap list"**, and providing the platform with a ranking of high-risk hosts for operational intervention.

### Worst-Case Scenario

The model achieves only **AUC 0.65–0.72**, coefficient directions are unstable or contradict expectations. The most typical failure mode is: `host_response_rate` is assigned an abnormally high weight by the model (even exceeding the rating), because in the dataset, response rate has many missing values, and when missing values are treated as a separate category, they happen to cluster in a group of hosts who "have never been asked questions but have high ratings"—the model learns the spurious correlation of "missing response rate = high probability of becoming Superhost." Another failure mode is: `calculated_host_listings_count` shows a negative coefficient, but after stratifying by `room_type`, the direction reverses (Simpson's paradox), because professional hosts mostly operate entire homes, and entire homes have different rating distributions than private rooms. The decision tree overfits, producing rules with too fine granularity (e.g., relying on a specific neighborhood combined with a specific property type), which cannot be generalized. In the worst case, the model's business recommendations will mislead hosts—for example, advising hosts to "prioritize increasing response rate" when in fact rating is the binding constraint. **The signal of this scenario is: when performing variable importance ranking in Dataiku, the top three features include some ID-type variable or a variable with extremely high missing rate**; the confusion matrix shows that the model almost predicts all samples as "non-Superhost" (because the positive class proportion is below 30% and no class balancing was done).

### How to Determine Which Scenario Is More Likely

After running the model in Dataiku, focus on three diagnostic points: (1) **Whether the ROC curve is significantly above the diagonal**, and whether the AUC gap between training and test sets is less than 0.05; (2) Whether the p-values of `review_scores_rating` and `host_response_rate` in the **logistic regression coefficient table** are both < 0.01, and whether the signs match expectations; (3) Whether the first splitting variable of the **decision tree** is rating rather than response rate or number of listings. If all three are satisfied, the result leans toward the best-case scenario; if any one fails, it needs to be addressed in the "improvement" phase.

---

## II. Model Interpretation (Business and Applied Perspective)

**Significant Variable 1: Overall Rating (`review_scores_rating`)**

This is the variable with the **largest marginal effect** in the model. In the San Francisco Bay Area study, "obtaining and maintaining excellent ratings" is by far the most important Superhost criterion, far exceeding the other three criteria. Airbnb's official threshold is 4.8, but the model will reveal that the marginal effect is not linear—the probability increase from 4.7 to 4.8 is much larger than from 4.8 to 4.9, because 4.8 is the qualification line. **Business translation**: For hosts hovering between 4.6 and 4.7, the top priority is to identify which sub-item (cleanliness, check-in, etc.) is dragging down the overall score, rather than spreading resources evenly.

**Significant Variable 2: Response Rate (`host_response_rate`)**

The model shows a significant positive effect, but its importance is lower than rating. Airbnb's official requirement is 90%, and research ranks responsiveness third. **Business translation**: Response rate is the Superhost metric most easily improved in the short term. Hosts using automated first-response tools can push response rate from 85% to 95%+ within a week, but the model will show that this alone is not enough to cross the Superhost threshold—it is a "necessary but not sufficient" condition. Platform operations teams can use this to design intervention strategies: first push response rate (low cost, quick results), then focus on rating improvement.

**Significant Variable 3: Number of Reviews (`number_of_reviews`)**

The model shows a positive effect but with diminishing marginal returns. New listings (reviews < 10) have extremely low predicted probabilities, but once review count exceeds approximately 40, the marginal contribution becomes negligible. **Business translation**: Airbnb's official requirement is "10+ stays or 100+ nights". The model confirms that the 10-stay threshold is indeed a hard constraint—hosts below this line cannot enter the Superhost candidate pool regardless of quality. The platform should provide a "new host pathway" that does not rely on review accumulation, such as a probationary badge after the first 5 five-star reviews.

**Significant Variable 4: Number of Host Listings (`calculated_host_listings_count`)**

The direction of this variable is the most likely to produce surprises. Studies have found that commercial hosts are **more likely** to obtain Superhost status, because they have systematic operational capabilities. But the model may also show the opposite direction, depending on the proportion of entire homes versus private rooms in the sample. **Business translation**: Single-listing hosts need to compensate for their lack of scale with "high rating + fast response"; multi-listing hosts should note that Superhost status is calculated at the **account level**—one poorly rated listing can block the badge for all other listings. The model's predicted probability can serve as a "portfolio health check" tool to identify which listing is the weakest link.

**Ten Key Insights (Model Output → Airbnb Strategy)**

1. **Rating is the most valuable lever**: A 0.1 rating increase is worth more than a 10 percentage point response rate increase—hosts should invest in cleaning and check-in experience first.
2. **Response rate is a "qualifier," not a "differentiator"**: After 90% is met, further increases have diminishing returns; resources should be shifted to rating.
3. **The 4.8 threshold has a cliff effect**: The model's probability curve steepens sharply around 4.8; hosts below this line should prioritize short-term interventions (e.g., proactively resolving negative reviews).
4. **New listings have an inherent disadvantage**: The 10-stay hard threshold means new listings cannot enter the Superhost track in their first quarter—the platform needs to design alternative trust signals.
5. **Account-level calculation amplifies risk**: For multi-listing hosts, the weakest listing determines the overall fate—the model output can be used for portfolio diagnostics.
6. **Missing values themselves carry information**: Listings with missing review scores are mostly new listings or low-activity listings; the model should treat "no reviews" and "bad reviews" differently.
7. **Superhost premium is real**: Research shows Superhosts earn 23%–28% more revenue than comparable non-Superhosts, and the model's predicted probability can serve as an estimation tool for hosts to quantify their "revenue at risk."
8. **Certification can partially offset bias**: Research finds that Black hosts gain more reservation growth from Superhost status (approximately 20%), and the model's fairness audit can test whether the predicted probabilities for different host groups are systematically low.
9. **Response rate automation is table stakes**: Hosts with response rates below 90% are high-risk churn targets; the platform can trigger automated training or tool recommendations.
10. **Decision tree rules are more actionable than probabilities**: Packaging rules like "rating > 4.85 + response rate > 95% + reviews > 40" as a host self-check tool has higher practical adoption than explaining logistic regression coefficients.

---

## III. Critical Conclusions and Automation/Product Development

**Critical Analysis: Limitations, Data Issues, Model Weaknesses**

The most fundamental limitation is **the subjectivity and saturation of rating data**. Airbnb ratings are extremely compressed—the vast majority of listings have ratings between 4.5 and 5.0, and guests' rating standards are influenced by listing expectations (a budget listing with 4.8 may actually have lower objective quality than a luxury listing with 4.7). The model treats `review_scores_rating` as a continuous variable, but its actual discriminative power is limited. Second, `license` has only 1 non-null value, `bathrooms` is entirely null, making it impossible to test regulatory compliance and physical amenities' impact on Superhost status. The 26% missing rate in `review_scores_*` is not random—new listings and low-activity listings are systematically missing, and treating missing as a separate category, while preserving samples, introduces selection bias. Additionally, the model's cross-sectional design cannot capture the **dynamic** nature of Superhost status—Airbnb reassesses quarterly with a 365-day rolling window, and a host's probability of "maintaining" versus "first obtaining" the badge may be driven by different factors.

**Improvement Directions: What Future Iterations Could Enhance**

First, introduce **text features**: perform sentiment scoring and topic extraction on `description` and `amenities` to identify which amenity combinations (e.g., workspace, parking) are associated with Superhost status. Second, use **temporal validation** instead of random splitting: train on data before 2020 and test on data from 2021 onward, to simulate the real-world "predicting the future" scenario. Third, address class imbalance: if the positive class proportion is below 30%, enable class weights or adjust the classification threshold, rather than using the default 0.5. Fourth, consider the **endogeneity** issue: Superhost status itself brings more bookings and reviews, which in turn affects rating and review count—a reverse causality exists. Instrumental variables or panel fixed effects can partially mitigate this. Fifth, incorporate **fairness auditing**: research confirms that certification programs such as Superhost have heterogeneous impacts across host groups, and the model should check whether the predicted probabilities for different host demographics (inferred from name or photo presence) show systematic deviations.

**Productization: How the Model Can Evolve into an Automated Dashboard or Recommender System**

The most direct product form is **"Superhost Readiness Dashboard"**: hosts input (or the system automatically pulls) response rate, rating trend, review count, and cancellation rate, and the model outputs (1) current predicted probability of becoming a Superhost, (2) the three metrics with the largest gaps and their required improvement magnitudes, (3) an "improvement priority ranking" based on marginal effects. In Dataiku, this can be implemented as an **API endpoint** or a **webapp**: the model is deployed as a scoring service, and the frontend displays gap bars and action recommendations. At the platform level, the model output can feed into an **operations intervention engine**: hosts with predicted probability below 0.3 and response rate below 90% enter the "high churn risk" list, automatically triggering response rate training push; hosts with predicted probability between 0.4–0.6 and rating constraint as the bottleneck enter the "near-threshold intervention" list, triggering cleaning standards or check-in process optimization recommendations. **Prototype suggestion**: Use Dataiku's webapp feature to build a two-page application—page one has the host input form, page two displays predicted probability and improvement recommendations—with the backend calling the trained logistic regression model. The limitation of this prototype is that it relies on static model weights and does not include real-time feedback loops; the future evolution direction is to introduce online learning, using each quarter's assessment results to automatically update model parameters.

---

**Appendix Note**: Model output screenshots (ROC curve, coefficient table, decision tree rules, confusion matrix) should be placed separately in the appendix, each annotated with the Dataiku experiment number and run date. In the main text, keep only the "three diagnostic points" judgment and the business interpretation of key variables.


In [ ]:
25% Translation of results to business insights, discussion of limitations, and future applications.                                                                (500 words)

In [ ]:
Q7. Model Interpretation (Business and Applied Perspective)
    Convert statistical outcomes into business meaning. 
    Explain what each significant variable implies in the real-world Airbnb context. 
    List up to 10 key insights connecting model outputs to Airbnb strategy or decision-making.

In [ ]:
Q8. Critical Conclusions & Automation/Product Development
    Maximum three paragraphs:
    • Critical analysis: limitations, data issues, model weaknesses.
    • Improvements: what could be enhanced in future iterations.
    • Productization: how the model could evolve into an automated dashboard or recommender.
    (Maybe develop a prototype also)

In [ ]:
Q9. Proposed Additional Data Science Use Case (implementation is not needed)
    Propose one additional Airbnb-related data science use case, 
    other than predicting whether a listing is successful.
    
    <!-- You are not required to build or execute this use case. -->
    Briefly explain:
    • Use case: What problem would you solve?
    • Business value: Why would this be useful?
    • User/stakeholder: Who would benefit from the solution?
    • Data required: Which existing Airbnb variables or additional data would be
    needed?
    • Possible analytical approach: What type of analysis or model could potentially
    be used?
    • Expected output: What would the final output look like?

In [ ]:
7. Propose an Additional Data Science Use Case: 
    Identify one additional Airbnb related business or analytical problem that could be addressed using the available datasets or suitable additional data. 
    Explain what the use case is, 
    who would benefit from it, 
    what data would be required, and 
    how it could potentially be approached using data science.

In [ ]:
Based on the dataset you provided and existing academic evidence, I recommend implementing the **"Superhost Trust Premium"** as an additional data science use case. This is distinct from the previous "Superhost prediction" task: the prediction task answers "will this host become a Superhost," while the premium analysis answers **"how much extra revenue does the Superhost badge actually bring, and for which types of hosts is this premium most significant."** Below is a breakdown by the six elements you requested.

---

### Use Case: Quantifying the "Trust Premium" of Superhost Badges

**What problem to solve**

The Superhost badge is Airbnb's most visible trust signal, but its economic value is not uniform. Studies show that in small cities in the United States, Superhost status significantly increases revenue, but its effect is smaller than the number of reviews and the number of photos. Airbnb officially claims that Superhosts earn 60% more than non-Superhosts, but this figure conflates causality (the badge brings revenue) with selection effect (high-quality hosts are more likely to become Superhosts). This use case aims to use observational data to **separate the net premium of the badge after controlling for other quality signals**, and identify **which market segments (by room type, neighborhood, host size) have the highest trust premium**.

**Business value**

For hosts: if the premium is concentrated in specific segments, hosts can determine whether it is "worth it" to invest resources to pursue the badge. For platforms: if the premium is weak or negative in certain segments, it indicates that the badge's signal validity is insufficient and the evaluation criteria need adjustment. For regulators: if the premium stems primarily from search ranking preferences rather than genuine quality differences, it may constitute a platform's self-preferencing behavior. Existing research has confirmed that the Superhost badge has a moderating effect on the relationship between review volume and revenue, indicating that trust signals do not operate independently but interact.

**Beneficiaries**

Airbnb platform operations and trust & safety teams (optimizing badge design), individual hosts and professional hosting management companies (investment decisions), short-term rental market analysts and urban policy researchers (understanding platform power).

**Data required**

Existing variables: `price` (needs cleaning), `host_is_superhost`, `room_type`, `property_type`, `neighbourhood_cleansed`, `number_of_reviews`, `review_scores_rating`, `calculated_host_listings_count`, `accommodates`, `availability_365`, `instant_bookable`.

Additional data (optional but valuable): historical snapshots of listing revenue (to construct panel data), listing search ranking position (to test the "exposure" channel), host tenure (`host_since` can serve as a proxy).

**Analytical approach**

Core method is **OLS regression + propensity score matching (PSM)**. First, use OLS to estimate the association between Superhost and revenue (proxied by `price × availability` or actual revenue), controlling for rating, review count, room type, neighborhood fixed effects, and host size. Then use PSM to match Superhosts with similar characteristics to non-Superhosts, reducing selection bias. If historical snapshot data is available, further use **difference-in-differences** to compare revenue changes for the same host before and after "acquiring" the badge. The academic literature already has a precedent using OLS and quantile regression to estimate the Superhost premium in Los Angeles, which can serve as a methodological benchmark.

**Expected output**

A **"Trust Premium Heatmap"**: rows are room types (entire home/private room/shared room), columns are neighborhood tiers (city center/suburbs), with values representing the net premium percentage of Superhost on revenue after controlling for other variables. Coupled with a summary table: "The Superhost premium is highest for entire-home listings in suburban neighborhoods (estimated +18%), but insignificant for private-room listings in city centers (+3%, p>0.1)." The final product can be a Dataiku dashboard where users select a neighborhood and room type to view the premium estimate and confidence interval for that segment.

---

### Critical Conclusions and Automation/Product Development

**Critical analysis**: The biggest threat to this use case is **endogeneity**—Superhosts may simply be better hosts to begin with, and their higher revenue may not be caused by the badge. Although PSM can alleviate this, it cannot eliminate unobserved confounders (such as host effort level, property quality). Second, `price` is a static snapshot price, not actual transaction price; using it to calculate revenue introduces measurement error. Additionally, missing values in `review_scores_rating` are non-random (new listings are more likely to be missing), and the matched sample may systematically exclude new hosts.

**Improvements**: If multiple time-point snapshots can be obtained (Inside Airbnb provides historical data), a **panel fixed effects model** should be used to control for time-invariant host heterogeneity. Alternatively, leverage Airbnb's quarterly assessment timing to construct a **regression discontinuity design (RDD)** based on the 4.8 rating threshold—comparing hosts just above and just below 4.8, their probability of obtaining the badge jumps discontinuously, thereby identifying the net effect of the badge. This is the most credible causal identification strategy for this question.

**Productization**: Evolve into a **"Badge ROI Calculator"**—hosts input their listing characteristics, and the system outputs "if you obtain Superhost status, based on your market segment's premium estimate, your expected annual revenue increase is $X; to achieve this, you need to increase your rating by Y points and response rate by Z percentage points." The backend calls the premium estimation model and PSM matching results. Further, it can be productized as an **automated monitoring dashboard** that tracks quarterly changes in premium across segments; if the premium in a segment continues to decline, it indicates that the badge is "inflating" and the platform needs to tighten evaluation criteria to maintain signal value.

---

### Why This Use Case Is More Valuable Than "Listing Density"

"Listing Density" is essentially a descriptive analysis (counting listings per neighborhood), with limited business actions: knowing density is high doesn't tell you what to do. The "Superhost Trust Premium," however, directly addresses **the core economic question of whether platform-designed trust mechanisms actually work**. It has clear causal identification challenges (endogeneity, selection bias) and corresponding solutions (PSM, RDD, panel FE), demonstrating data science depth, and its output can directly inform host decisions and platform design. Existing evidence already shows that the Superhost badge has a significant moderating effect on the relationship between review volume and revenue, indicating that trust signals interact—this is a direction worthy of deeper investigation.